In [5]:
from datetime import datetime
from pathlib import Path
from urllib.error import URLError
from zoneinfo import ZoneInfo
import pandas as pd

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
# Options: "all.csv", "sp500.csv", "top_100.csv", "top_200.csv", "top_50.csv"
TICKER_SET: str = "top_50.csv"

# Toggle whether to purge old CSV files in 'data/ticker_data/' before saving a new pull
CLEAR_EXISTING_DATA: bool = True

# Base repository URL
BASE_URL: str = "https://raw.githubusercontent.com/zyhe16/top-us-stock-tickers/main/tickers/"
DATA_URL: str = f"{BASE_URL}{TICKER_SET}"


def clear_ticker_data_folder(folder_path: Path) -> None:
    """Deletes all CSV files in the target directory."""
    csv_files = list(folder_path.glob("*.csv"))
    if not csv_files:
        return

    print(f"Clearing {len(csv_files)} existing file(s) from {folder_path}...")
    for file in csv_files:
        try:
            file.unlink()
            print(f" Deleted: {file.name}")
        except Exception as e:
            print(f" Failed to delete {file.name}: {e}")


def fetch_ticker_data() -> None:
    # 1. Determine current date in US Eastern Time (DD-MM-YY)
    us_eastern_time = datetime.now(ZoneInfo("America/New_York"))
    date_str = us_eastern_time.strftime("%d-%m-%y")

    # 2. Dynamically find script_dir whether running as a .py script or in Jupyter/REPL
    try:
        script_dir = Path(__file__).resolve().parent
    except NameError:
        # Fallback to Current Working Directory if running interactively
        script_dir = Path.cwd()

    # If the script lives in 'src/', step up one level to project root; otherwise use cwd
    if script_dir.name == "src":
        project_root = script_dir.parent
    else:
        project_root = script_dir

    # # 2. Target the data folder
    # output_dir = project_root / "data" / "ticker_data"
    # output_dir.mkdir(parents=True, exist_ok=True)


    # # 2. Locate the output directory dynamically
    # script_dir = Path(__file__).resolve().parent
    # project_root = script_dir.parent
    output_dir = project_root / "data" / "ticker_data"

    # Create 'data/ticker_data/' if it doesn't exist yet
    output_dir.mkdir(parents=True, exist_ok=True)

    # 3. Construct the target file path using the constant prefix
    file_prefix = Path(TICKER_SET).stem  # e.g., 'top_50.csv' -> 'top_50'
    file_name = f"{date_str}_{file_prefix}_ticker_data.csv"
    file_path = output_dir / file_name

    # 4. Check if the target file already exists
    if file_path.exists():
        print(f"Latest ticker data for '{TICKER_SET}' has already been pulled ({file_name})")
        return

    # 5. If the file doesn't exist and CLEAR_EXISTING_DATA is True, clear old CSVs
    if CLEAR_EXISTING_DATA:
        clear_ticker_data_folder(output_dir)

    # 6. Try-Except block around the remote network request and parsing
    try:
        print(f"Fetching {TICKER_SET} from {DATA_URL}...")
        df = pd.read_csv(DATA_URL)

        # Validate that the DataFrame is not empty
        if df.empty:
            raise ValueError("Downloaded dataset contains no data.")

        # Save to local directory
        df.to_csv(file_path, index=False)
        print(f"Successfully fetched and saved data to: {file_path}")

    except (URLError, pd.errors.ParserError, pd.errors.EmptyDataError) as e:
        print(f"Failed to fetch ticker data due to network or parsing error: {e}")
    except ValueError as ve:
        print(f"Validation error: {ve}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


# if __name__ == "__main__":
fetch_ticker_data()

Clearing 2 existing file(s) from c:\Users\Andre\Documents\git\ai_financial_advisor\data\ticker_data...
 Deleted: 07-08-26_top_50_ticker_data.csv
 Deleted: 08-08-26_top_50_ticker_data.csv
Fetching top_50.csv from https://raw.githubusercontent.com/zyhe16/top-us-stock-tickers/main/tickers/top_50.csv...
Successfully fetched and saved data to: c:\Users\Andre\Documents\git\ai_financial_advisor\data\ticker_data\09-08-26_top_50_ticker_data.csv
